In [ ]:
from matplotlib import pyplot as plt
import numpy as np
import pandas as pd
import os
import seaborn as sns

if os.getcwd().endswith('notebooks'):
    os.chdir("..")

from src.pheno import double_logistic

In [ ]:
pheno = pd.read_parquet("phenocam_phenology_model.parquet")

In [ ]:
pheno.columns

In [ ]:
dbf = pheno.query("veg_type == 'Deciduous forest'")
doy_x = np.linspace(-100, 450, num=200)
curves = np.zeros((dbf.shape[0], doy_x.shape[0]))


for i, (idx, row) in enumerate(pheno.query("veg_type == 'Deciduous forest'").iterrows()):
    curves[i] = double_logistic(doy_x, row["transition_slope_rising"], row["transition_slope_falling"], row["doy_transition_50_rising"], row["doy_transition_50_falling"])

In [ ]:
fig = plt.figure(figsize=(10, 4), dpi=150)

plt.plot(doy_x, curves.T, alpha=0.1)
plt.grid()
plt.xlabel("Day of year")
plt.ylabel("Relative activity")
plt.title(f"Deciduous forests, {curves.shape[0]} transitions", loc="left")
plt.savefig("figures/dbf_transitions.png")
plt.show()

In [ ]:
pheno["doy_transition_50_rising_clip"] = pheno["doy_transition_50_rising"] % 365

fit = np.polyfit(pheno["tmax"], pheno["doy_transition_50_rising_clip"], 1)

fig, ax = plt.subplots(figsize=(7, 4))

sns.scatterplot(data=pheno, ax=ax, x="tmax", y="doy_transition_50_rising_clip", hue="veg_type", s=(6**2))
plt.gca().axline((0, fit[1]), slope=fit[0], color="black", linestyle="dashed")
plt.xlabel("Mean annual Tmax (deg C)")
plt.ylabel("Greenup DOY")
plt.grid()
#plt.savefig("figures/greenup_doy_tmax.png")
plt.show()

In [ ]:
from src.model import CircularTransformer

In [ ]:
tformer = CircularTransformer(cycle_length=365).fit(pheno["doy_transition_50_rising"])
pheno["doy_transition_50_rising_tform"] = tformer.transform(pheno["doy_transition_50_rising"])

In [ ]:
fit = np.polyfit(pheno["tmax"], pheno["doy_transition_50_rising_tform"], 1)

fig, ax = plt.subplots(figsize=(7, 4))

sns.scatterplot(data=pheno, ax=ax, x="tmax", y="doy_transition_50_rising_tform", hue="veg_type", s=(6**2))
plt.gca().axline((0, fit[1]), slope=fit[0], color="black", linestyle="dashed")
plt.xlabel("Mean annual Tmax (deg C)")
plt.ylabel("tan(Greenup DOY)")
plt.grid()
plt.savefig("figures/greenup_doy_tmax_tform.png")
plt.show()

PLSR model diagnostics

In [ ]:
import skops.io as sio

model = sio.load(file="phenocam_monthly_plsr.skops", trusted=sio.get_untrusted_types(file="phenocam_monthly_plsr.skops"))
climate = pd.read_parquet("phenocam_site_in_usa_climate.parquet")
climate.columns = ["year", "yday", "prcp", "tmax", "tmin", "vp", "site"]
climate["date"] = pd.to_datetime(climate['year'], format='%Y') + pd.to_timedelta(climate['yday'] - 1, unit='D')

climate_monthly = climate\
    .groupby(["site", "year", climate["date"].dt.month])[["prcp", "tmax"]]\
    .agg({"prcp": "sum", "tmax": "mean"})\
    .reset_index()\
    .assign(month = lambda x: "month" + x.date.astype(str))\
    .drop(columns="date")\
    .pivot(index=["site", "year"], columns="month")

climate_monthly.columns = ['_'.join(col).strip() for col in climate_monthly.columns.values]

climate_monthly.head()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
im = ax.matshow(climate_monthly.corr(), vmin=-1, vmax=1, cmap="RdBu_r")
plt.yticks([6, 18], ["Monthly Prcp", "Monthly Tmax"])
plt.xticks([6, 18], ["Monthly prcp", "Monthly Tmax"])
plt.colorbar(im, label="Correlation")
plt.savefig("figures/monthly_corrmtx.png")
plt.show()

In [ ]:
climate_monthly_tform = model.regressor_.transform(climate_monthly)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))

corr_matrix = np.corrcoef(climate_monthly_tform, rowvar=False)

labs = [f"Feat. {x+1}" for x in range(corr_matrix.shape[0])]

plt.yticks([0, 1, 2, 3], labs)
plt.xticks([0, 1, 2, 3], labs)

im = ax.matshow(corr_matrix, vmin=-1, vmax=1, cmap="RdBu_r")
plt.colorbar(im, label="Correlation")
plt.savefig("figures/plsr_feature_corrmtx.png")
plt.show()

In [ ]:
import re

pattern = re.compile(r"(\d+)$")

# Coefficients are on the scale of days per standard deviation of each input feature
scaled_plsr_coefs = model.regressor_.coef_ * model.regressor_._x_std * (365 / (2*np.pi))

# Reorder so that we can plot months sequentially
month = np.array([int(re.findall(pattern, x)[0]) for x in model.feature_names_in_])
kind  = [x.split("_")[0] for x in model.feature_names_in_]


feature_names_sort = model.feature_names_in_[np.lexsort((month, kind))]
scaled_plsr_coefs_sort = scaled_plsr_coefs[0, np.lexsort((month, kind))]

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True, sharey=True)

x = np.arange(12)
months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

# Horizontal grid marks
ax1.grid(axis="y")
ax2.grid(axis="y")

ax1.bar(x, scaled_plsr_coefs_sort[:12])
ax2.bar(x, scaled_plsr_coefs_sort[12:])

# For symmetrical y-axis
ylim = ax1.get_ylim()
vmax = max(abs(ylim[0]), abs(ylim[1]))
ax1.set_ylim(-vmax, vmax)
ax2.set_ylim(-vmax, vmax)

ax2.set_xticks(x, months)

ax1.set_title("Precipitation")
ax2.set_title("Max temperature")

fig.supylabel("$\\Delta$ Greenup DOY / S.D.")

plt.savefig("figures/plsr_coefficient_plot.png")
plt.show()